In [ ]:
import rasterio
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings("ignore")

class_labels = ['Spruce', 'Birch', 'Mix', 'Pine', 'Aspen',]

def get_metrics(y_pred, y_test):
    y_test = np.array(y_test)
    y_pred = np.array(y_pred)

    print("Classification Report:\n", classification_report(y_test, 
                                                            y_pred, 
                                                            labels=np.arange(len(class_labels)), 
                                                            target_names=class_labels,))

    plt.figure(figsize=(10, 6))
    plt.hist(y_test, label='True Labels', alpha=0.5, bins=30)
    plt.hist(y_pred, label='Predicted Labels', alpha=0.5, bins=30)
    plt.xlabel('Class Labels')
    plt.ylabel('Frequency')
    plt.legend()
    plt.title('True vs Predicted Class Labels')
    plt.show()

def plot_full_map_prediction(df_orig, y_pred_full, species):
    species = np.squeeze(species)
    y_pred_reshaped = np.full_like(species, np.nan, dtype=float)
    y_pred_reshaped.ravel()[df_orig.dropna().index] = y_pred_full
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1 = axes[0]
    im1 = ax1.imshow(species, interpolation='nearest')
    ax1.set_title("Original Species Map")
    
    ax2 = axes[1]
    im2 = ax2.imshow(y_pred_reshaped, interpolation='nearest')
    ax2.set_title("Predicted Species Map")
    
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im1, cax=cax)
    cbar.set_label("Species Class")
    cbar.set_ticks(np.arange(len(['None'] + class_labels)))
    cbar.set_ticklabels(['None'] + class_labels)
    
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    plt.show()

### Read data

In [ ]:
y_src = rasterio.open('data/species_882_2021.tiff')

species = y_src.read(1)

In [ ]:
plt.imshow(species)
plt.colorbar()

In [ ]:
x_src = rasterio.open('data/source_882_2021.tiff')

In [ ]:
bands = x_src.read()
bands.shape

### Preprocessing

In [ ]:
species = np.expand_dims(species, axis=0)
data = np.concatenate( (bands, species), axis=0)

In [ ]:
df = data.reshape([15, data.shape[1] * data.shape[2]])

In [ ]:
df = pd.DataFrame(
    df.T,
    columns=["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09", "B11","B12","SCL","CLM", "species"])
df = df.replace(65535.0, np.nan)

df_orig = df.copy()

df.dropna(inplace=True)

In [ ]:
features = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09',
       'B11', 'B12', 'SCL', 'CLM']

scaler = StandardScaler()
data = scaler.fit_transform(df[features].copy())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data, df.species, test_size=0.33, random_state=42)

### Models

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
logreg_model = LogisticRegression()
logreg_model.fit(X_train, y_train)

In [ ]:
y_pred = logreg_model.predict(X_test)

In [ ]:
get_metrics(y_pred, y_test)

In [ ]:
y_pred_full = logreg_model.predict(data)

plot_full_map_prediction(df_orig, y_pred_full, species)
